Load the dataset 

In [2]:
import pandas as pd
df = pd.read_csv('reviews.csv')
df.head()
df.shape
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [3]:
df['Score'].value_counts()

Score
5    363122
4     80655
1     52268
3     42640
2     29769
Name: count, dtype: int64

In [4]:
df['Score'].value_counts(normalize=True) # % get percentage 

Score
5    0.638789
4    0.141885
1    0.091948
3    0.075010
2    0.052368
Name: proportion, dtype: float64

In [5]:
df_copy = df.copy()

In [6]:
df_copy.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [7]:
# Now we have to map the Score to Sentiment using (.map)
dict = {1:0,2:0,4:1,5:1}
df_copy['Sentiment']= df_copy['Score'].map(dict)
df_copy.head()
df_copy['Sentiment'].value_counts()


Sentiment
1.0    443777
0.0     82037
Name: count, dtype: int64

In [8]:
# since we are not considering 3 for this baseline which include Nan values (applying dropna)
df_copy = df_copy.dropna(subset=['Sentiment'])
df_copy['Sentiment'].isnull().any()


np.False_

In [9]:
type(df_copy)

pandas.core.frame.DataFrame

In [10]:
df_copy['Text'].sample(5, random_state=0)
pd.set_option("display.max_colwidth", None)

In [11]:
df_copy['Text'].sample(5, random_state=0)

112249                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     We had pretty much given up on GF pasta until we had this in a restaurant.  It's really good and doesn't disintegrate when you toss it with sauce.  My celiac

In [16]:
# Remove the leading and trailing white space in the DataFrame 
df_copy['Text'] = df_copy['Text'].str.strip()

In [17]:
df_copy['Text'].sample(5, random_state=0)

112249                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     We had pretty much given up on GF pasta until we had this in a restaurant.  It's really good and doesn't disintegrate when you toss it with sauce.  My celiac

In [24]:
repr(df_copy["Text"].iloc[0])

"'I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.'"

In [40]:
# Lets define Feature and Target variables 
X = df_copy['Text']
Y = df_copy['Sentiment']

In [41]:
# Apply Trin test split 
# Apply stratified split since the dataset is imbalanced and we need too ensure the split ratio is same 
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0,stratify=Y)

In [ ]:
# To ensure the split happend correctly 
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(420651,) (420651,)
(105163,) (105163,)


In [44]:
# Now we want to ensure the stratification actually worked in both train and test dataset 
Y_train.value_counts(normalize=True)
Y_test.value_counts(normalize=True)

Sentiment
1.0    0.843985
0.0    0.156015
Name: proportion, dtype: float64

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000).fit(X_train)

In [46]:
tf

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [49]:
train_tf= tf.transform(X_train)

In [50]:
test_tf= tf.transform(X_test)

In [53]:
# Checking the size of the transformed matrics 
print(train_tf.shape)
print(test_tf.shape)

(420651, 5000)
(105163, 5000)


In [62]:
# Ensure the number of features and target variables are equal
print(train_tf.shape)
print(Y_train.shape)

(420651, 5000)
(420651,)


In [84]:
# Apply Logistic regression 
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5)
logreg.fit(train_tf, Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.5
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [85]:
y_pred = logreg.predict(test_tf)

In [86]:
# Find the accuracy of the prediction 
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
accuracy_score(Y_test,y_pred)

0.902579804684157

We have obtained 93% of accuracy 

In [87]:
print(confusion_matrix(Y_test,y_pred))

[[14961  1446]
 [ 8799 79957]]


In [88]:
print(classification_report(Y_test,y_pred))


              precision    recall  f1-score   support

         0.0       0.63      0.91      0.74     16407
         1.0       0.98      0.90      0.94     88756

    accuracy                           0.90    105163
   macro avg       0.81      0.91      0.84    105163
weighted avg       0.93      0.90      0.91    105163



In [ ]:
# Apply model comparison 
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

svm_model = LinearSVC(class_weight='balanced', C=0.5)
svm_model.fit(train_tf, Y_train)

y_pred_svm = svm_model.predict(test_tf)

print(classification_report(Y_test, y_pred_svm))

              precision    recall  f1-score   support

         0.0       0.64      0.91      0.75     16407
         1.0       0.98      0.91      0.94     88756

    accuracy                           0.91    105163
   macro avg       0.81      0.91      0.85    105163
weighted avg       0.93      0.91      0.91    105163

